# 05 — ReAct Agent 工具调用

**来源:** [菜鸟教程 — LangGraph 入门教程](https://www.runoob.com/ai-agent/langgraph-quick-start.html)

ReAct（Reason + Act）是最常见的 Agent 模式：LLM 思考 → 选择工具 → 执行工具 → 观察结果 → 继续思考。

```mermaid
graph LR
    START --> Agent
    Agent -- "有工具调用" --> Tools
    Tools -- "执行结果" --> Agent
    Agent -- "完成" --> END
```

### 定义工具
使用 `@tool` 装饰器定义 Agent 可用的工具。

> **安全警告：** `calculate` 工具使用 AST 解析替代 `eval()`，避免执行任意 Python 代码的安全风险。

In [ ]:
from dotenv import load_dotenv
load_dotenv()

import os
import ast
import operator
from langgraph.graph import StateGraph, MessagesState, START, END
from langgraph.prebuilt import ToolNode, tools_condition
from langchain_core.tools import tool
from langchain_core.messages import HumanMessage
from IPython.display import Image, display


@tool
def search_web(query: str) -> str:
    """搜索网络获取最新信息。"""
    return f"关于 '{query}' 的搜索结果：这是模拟的搜索结果..."


@tool
def calculate(expression: str) -> str:
    """计算数学表达式。使用 AST 安全解析，避免 eval() 风险。"""
    ops = {
        ast.Add: operator.add,
        ast.Sub: operator.sub,
        ast.Mult: operator.mul,
        ast.Div: operator.truediv,
        ast.Pow: operator.pow,
        ast.USub: operator.neg,
    }

    def safe_eval(node):
        if isinstance(node, ast.Expression):
            return safe_eval(node.body)
        elif isinstance(node, ast.Constant):
            return node.value
        elif isinstance(node, ast.BinOp):
            left = safe_eval(node.left)
            right = safe_eval(node.right)
            return ops[type(node.op)](left, right)
        elif isinstance(node, ast.UnaryOp):
            operand = safe_eval(node.operand)
            return ops[type(node.op)](operand)
        else:
            raise ValueError(f"不支持的表达式类型: {type(node)}")

    try:
        tree = ast.parse(expression, mode='eval')
        result = safe_eval(tree)
        return f"计算结果: {expression} = {result}"
    except Exception as e:
        return f"计算错误: {str(e)}"


@tool
def get_weather(city: str) -> str:
    """获取指定城市的天气信息。"""
    return f"{city} 今日天气：晴，温度 22°C，湿度 60%"


tools = [search_web, calculate, get_weather]

# 初始化 LLM 并绑定工具
from langchain_deepseek import ChatDeepSeek

llm = ChatDeepSeek(model='deepseek-v4-pro', temperature=0)
llm_with_tools = llm.bind_tools(tools)


def agent_node(state: MessagesState) -> dict:
    """Agent 节点：调用 LLM 进行推理"""
    response = llm_with_tools.invoke(state["messages"])
    return {"messages": [response]}


# 构建 ReAct Agent 图
tool_node = ToolNode(tools)

builder = StateGraph(MessagesState)
builder.add_node("agent", agent_node)
builder.add_node("tools", tool_node)
builder.add_edge(START, "agent")
builder.add_conditional_edges("agent", tools_condition)
builder.add_edge("tools", "agent")

graph = builder.compile()

print("ReAct Agent 构建完成")
print("可用工具:", [t.name for t in tools])
display(Image(graph.get_graph().draw_mermaid_png()))
